In [3]:
! pip install aubio
! pip install tensorflow
import aubio
import numpy as np
import os
import csv
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split
from scipy.io.wavfile import write

# Extract pitches from WAV file using aubio
def get_chords(wav_file):
    samplerate = 44100
    win_s = 4096  # FFT window size
    hop_s = 512   # Hop size

    try:
        s = aubio.source(wav_file, samplerate, hop_s)
        o = aubio.pitch("yin", win_s, hop_s, samplerate)
        o.set_unit("midi")
        o.set_tolerance(0.5)

        pitches = []
        while True:
            samples, read = s()
            pitch = o(samples)[0]
            pitches.append(pitch)
            if read < hop_s:
                break
    except Exception as e:
        print(f"Error processing {wav_file}: {e}")
        return np.array([])

    return np.array(pitches)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.0/479.0 kB 11.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for aubio: filename=aubio-0.4.9-cp311-cp311-linux_x86_64.whl size=421851 sha256=78ef23ffc11debc544ef3c20e1808a728c0ff9ddf44072744eb80c2b9f804d00
  Stored in directory: /root/.cache/pip/wheels/61/2b/91/998862d8e1f1e620c1d81c57d0bbe1cd5fc09f66563496db3f
Successfully built aubio


In [4]:
# Convert MIDI pitches to audio waveform
def pitches_to_audio(pitches, hop_s=512, samplerate=44100):
    t = np.arange(len(pitches) * hop_s) / float(samplerate)
    signal = np.zeros_like(t)
    for i, pitch in enumerate(pitches):
        if pitch > 0:
            freq = 440.0 * (2.0 ** ((pitch - 69.0) / 12.0))
            start = i * hop_s
            end = start + hop_s
            if end > len(t):
                end = len(t)
            signal[start:end] += 0.5 * np.sin(2 * np.pi * freq * t[start:end])
    signal = (signal * 32767 / np.max(np.abs(signal))).astype(np.int16)
    return signal



In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Load audio files and extract pitch sequences
audio_folder = "/content/drive/MyDrive/Audio_Files"
all_pitches = []
labels = []

for subfolder in ["Minor", "Major"]:
    subfolder_path = os.path.join(audio_folder, subfolder)
    for filename in os.listdir(subfolder_path):
        if filename.endswith(".wav"):
            file_path = os.path.join(subfolder_path, filename)
            if os.path.exists(file_path):
                print(f"Processing: {filename} (from {subfolder})")
                pitches = get_chords(file_path)
                all_pitches.append(pitches)
                labels.append(0 if subfolder == "Minor" else 1)
            else:
                print(f"File not found: {file_path}")


Processing: Minor_147.wav (from Minor)
Processing: Minor_142.wav (from Minor)
Processing: Minor_113.wav (from Minor)
Processing: Minor_123.wav (from Minor)
Processing: Minor_148.wav (from Minor)
Processing: Minor_14.wav (from Minor)
Processing: Minor_108.wav (from Minor)
Processing: Minor_152.wav (from Minor)
Processing: Minor_110.wav (from Minor)
Processing: Minor_133.wav (from Minor)
Processing: Minor_117.wav (from Minor)
Processing: Minor_13.wav (from Minor)
Processing: Minor_101.wav (from Minor)
Processing: Minor_127.wav (from Minor)
Processing: Minor_154.wav (from Minor)
Processing: Minor_112.wav (from Minor)
Processing: Minor_134.wav (from Minor)
Processing: Minor_125.wav (from Minor)
Processing: Minor_109.wav (from Minor)
Processing: Minor_116.wav (from Minor)
Processing: Minor_151.wav (from Minor)
Processing: Minor_144.wav (from Minor)
Processing: Minor_145.wav (from Minor)
Processing: Minor_118.wav (from Minor)
Processing: Minor_107.wav (from Minor)
Processing: Minor_146.wav (

In [8]:
# Save extracted pitch data to CSV
csv_output_path = "extracted_pitches.csv"
with open(csv_output_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    for sequence in all_pitches:
        writer.writerow(sequence)
print(f"Pitch data saved to {csv_output_path}")


Pitch data saved to extracted_pitches.csv


In [9]:
# Pad sequences
max_length = max(len(seq) for seq in all_pitches)
all_pitches_padded = pad_sequences(all_pitches, maxlen=max_length, padding='post', dtype='float32')

# Create NumPy arrays
all_pitches_array = np.array(all_pitches_padded)
labels_array = np.array(labels)



In [10]:
# Normalize the data
all_pitches_array = all_pitches_array / 127.0  # MIDI values max at 127

# Split the dataset
x_train, x_test = train_test_split(all_pitches_array, test_size=0.2, random_state=42)


In [11]:

# Define VAE encoder-decoder architecture
input_dim = x_train.shape[1]
latent_dim = 32

inputs = Input(shape=(input_dim,))
encoded = Dense(128, activation='relu')(inputs)
encoded = Dense(latent_dim, activation='relu')(encoded)

# Decoder
decoded = Dense(128, activation='relu')(encoded)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

# Autoencoder model
autoencoder = Model(inputs, decoded)
autoencoder.compile(optimizer='adam', loss='mse')


In [12]:
# Train the model
autoencoder.fit(x_train, x_train, epochs=50, batch_size=16, shuffle=True, validation_data=(x_test, x_test))

# Encode and decode test data
reconstructed = autoencoder.predict(x_test)

Epoch 1/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - loss: 0.0497 - val_loss: 0.0214
Epoch 2/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0206 - val_loss: 0.0135
Epoch 3/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0129 - val_loss: 0.0100
Epoch 4/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0100 - val_loss: 0.0088
Epoch 5/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0085 - val_loss: 0.0081
Epoch 6/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0078 - val_loss: 0.0074
Epoch 7/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0072 - val_loss: 0.0068
Epoch 8/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0072 - val_loss: 0.0064
Epoch 9/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0063 - val_loss: 0.0061
Epoch 10/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0060 - val_loss: 0.0058
Epoch 11/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0058 - val_loss: 0.0055
Epoch 12/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0053 - val_l

In [13]:
# Convert predictions back to MIDI values
reconstructed_midi = reconstructed * 127.0

# Convert first reconstructed sample to audio
reconstructed_audio = pitches_to_audio(reconstructed_midi[0])
write("reconstructed_output.wav", 44100, reconstructed_audio)

print("\nReconstructed audio saved as 'reconstructed_output.wav'")


Reconstructed audio saved as 'reconstructed_output.wav'
